# Phase 3: LSTM Training & Evaluation (Refactored)

Key changes from previous version:
- **Input**: 28 features (14 means + 14 stds, sub-minute 30s aggregation)
- **Output hierarchy**: `Training_Results/Window_Size_{ws}/Fold_{fold}/`
- **Diagnostic tracking**: pre-training sanity check + prediction distribution logging
- **OOM prevention**: explicit tensor deletion + `gc.collect()` + CUDA cache clear after each fold
- **Plotting**: full life-cycle X-axis from absolute minute 0

In [ ]:
import gc
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.color": "#bfbfbf",
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
})
sns.set_palette("husl")

## 1. Configuration

In [ ]:
DATASET_DIR = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\LSTM_Inputs"
RESULTS_DIR = r"D:\Proyek Dosen\Riset Bearing\Training_Results"

WINDOW_SIZES = [10, 20, 30]
FOLDS = 5

# 14 extracted features x (mean + std) = 28 per timestep
NUM_FEATURES = 28

# Training hyperparameters
EPOCHS = 60
BATCH_SIZE = 128
LEARNING_RATE = 0.0005
WEIGHT_DECAY = 1e-5
HIDDEN_DIM = 128
NUM_LAYERS = 2
NOISE_STD = 0.0    # Disabled per preprocessing audit
DROPOUT = 0.1      # Halved from 0.2

# Each CSV row = 0.5 physical minutes (sub-minute 30s hop)
SUBMINUTE_HOP = 0.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computation Device: {DEVICE}")

## 2. Model Definition

Vanilla LSTM with no output activation (avoids Dying ReLU / output clamping).

In [ ]:
class RobustLSTM(nn.Module):
    def __init__(
        self,
        input_size: int = 28,
        hidden_dim: int = 128,
        num_layers: int = 2,
        output_size: int = 1,
        dropout: float = 0.1,
        noise_std: float = 0.0,
    ):
        super(RobustLSTM, self).__init__()
        self.noise_std = noise_std
        self.rnn = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_dim, output_size)  # No activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training and self.noise_std > 0.0:
            x = x + torch.randn_like(x) * self.noise_std
        out, _ = self.rnn(x)
        out = self.dropout(out[:, -1, :])
        return self.linear(out)

## 3. Utility Functions

In [ ]:
def calculate_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def reconstruct_3d(flat: np.ndarray, window_size: int, n_feat: int) -> np.ndarray:
    """Reshape (samples, ws*feat) -> (samples, ws, feat)."""
    return flat.reshape(-1, n_feat, window_size).transpose(0, 2, 1)


def run_sanity_check(feature_cols, x_train, x_val, log_path):
    """Log feature order and global min/max to verify MinMax scaling held."""
    report = {
        "feature_count": len(feature_cols),
        "feature_order": feature_cols,
        "train_global_min": float(x_train.min()),
        "train_global_max": float(x_train.max()),
        "val_global_min": float(x_val.min()),
        "val_global_max": float(x_val.max()),
        "train_any_nan": bool(np.isnan(x_train).any()),
        "val_any_nan": bool(np.isnan(x_val).any()),
        "train_any_inf": bool(np.isinf(x_train).any()),
        "val_any_inf": bool(np.isinf(x_val).any()),
    }
    for k, v in report.items():
        if k != "feature_order":
            print(f"[SANITY] {k:<25}: {v}")
    with open(log_path, "w", encoding="utf-8") as fh:
        json.dump(report, fh, indent=2)
    return report


def log_pred_distribution(pred_hi, log_dict, fold, ws):
    """Detect mean prediction collapse: Std < 0.05 triggers a warning."""
    stats = {
        "fold": fold, "window_size": ws,
        "pred_min": float(pred_hi.min()),
        "pred_max": float(pred_hi.max()),
        "pred_mean": float(pred_hi.mean()),
        "pred_std": float(pred_hi.std()),
    }
    collapsed = stats["pred_std"] < 0.05
    stats["collapse_warning"] = collapsed
    suffix = "  <<< COLLAPSE DETECTED >>>" if collapsed else ""
    print(
        f"[PRED DIST] Fold {fold} WS {ws} | "
        f"Min={stats['pred_min']:.4f} Max={stats['pred_max']:.4f} "
        f"Mean={stats['pred_mean']:.4f} Std={stats['pred_std']:.4f}{suffix}"
    )
    log_dict[f"fold_{fold}_pred_dist"] = stats
    return log_dict


def plot_health_index(smoothed_df, ws, fold, output_dir):
    """
    Full life-cycle plot: X-axis from absolute minute 0.
    True_HI = solid blue. Smoothed_Predicted_HI = dashed red.
    The healthy plateau (True_HI == 1.0) is shaded.
    """
    bearings = (
        smoothed_df["Bearing_ID"].unique()
        if "Bearing_ID" in smoothed_df.columns
        else ["Unknown"]
    )
    fig, axes = plt.subplots(len(bearings), 1, figsize=(12, 4 * len(bearings)), squeeze=False)

    for idx, bearing in enumerate(sorted(bearings)):
        ax = axes[idx, 0]
        b_df = smoothed_df[smoothed_df["Bearing_ID"] == bearing].copy()

        if "Step_Index" in b_df.columns:
            b_df = b_df.sort_values("Step_Index")
            x_min = b_df["Step_Index"].values * SUBMINUTE_HOP
        elif "Original_Minute" in b_df.columns:
            b_df = b_df.sort_values("Original_Minute")
            x_min = b_df["Original_Minute"].values
        else:
            b_df = b_df.reset_index(drop=True)
            x_min = b_df.index.values * SUBMINUTE_HOP

        true_hi = b_df["True_HI"].values
        pred_col = "Smoothed_Predicted_HI" if "Smoothed_Predicted_HI" in b_df.columns else "Predicted_HI"
        pred_hi = b_df[pred_col].values

        ax.plot(x_min, true_hi, color="#1f77b4", linewidth=2.0, label="Ground Truth HI", zorder=3)
        ax.plot(x_min, pred_hi, color="#d62728", linewidth=1.8,
                linestyle="--", label="LSTM Prediction (EMA Smoothed)", zorder=4)

        if (true_hi == 1.0).any():
            plateau_end = x_min[true_hi == 1.0][-1]
            ax.axvspan(x_min[0], plateau_end, alpha=0.07, color="#1f77b4", label="Healthy Phase")

        ax.set_title(f"LSTM HI Prediction  |  WS={ws}  Fold={fold}  |  {bearing}", pad=8)
        ax.set_xlabel("Operating Time (Minutes)")
        ax.set_ylabel("Health Index (HI)")
        ax.set_xlim(left=0)
        ax.set_ylim(-0.05, 1.10)
        ax.legend(loc="upper right", frameon=True, framealpha=0.9, edgecolor="black")

    plt.tight_layout()
    fpath = os.path.join(output_dir, f"hi_plot_ws{ws}_fold{fold}.png")
    plt.savefig(fpath, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[PLOT] Saved: {fpath}")

## 4. Training & Evaluation Loop

For each window size and fold: sanity check → 3D reconstruct → train → batched inference → distribution log → EMA smooth → metrics → save artifacts → **explicit OOM cleanup**.

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
summary_metrics = []

print("=" * 70)
print(f"LSTM PIPELINE | Device={DEVICE} | Features={NUM_FEATURES}")
print(f"HiddenDim={HIDDEN_DIM} Layers={NUM_LAYERS} Epochs={EPOCHS} Batch={BATCH_SIZE}")
print("=" * 70)

META_COLS = ["Health_Index", "Target_RUL", "Bearing_ID",
             "Change_Point", "Original_Minute", "Minute", "Step_Index"]

for ws in WINDOW_SIZES:
    print(f"\n{'='*70}")
    print(f"  WINDOW SIZE: {ws}  ({ws * SUBMINUTE_HOP:.1f} physical minutes context)")
    print(f"{'='*70}")

    ws_data_dir = os.path.join(DATASET_DIR, f"window_size_{ws}")
    ws_results_dir = os.path.join(RESULTS_DIR, f"Window_Size_{ws}")
    os.makedirs(ws_results_dir, exist_ok=True)

    fold_metrics = []
    ws_all_preds = pd.DataFrame()
    diag_log = {"window_size": ws}

    for fold in range(1, FOLDS + 1):
        fold_dir = os.path.join(ws_results_dir, f"Fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)
        print(f"\n  --- Fold {fold} ---")

        train_csv = os.path.join(ws_data_dir, f"processed_train_fold{fold}.csv")
        val_csv = os.path.join(ws_data_dir, f"processed_val_fold{fold}.csv")
        if not os.path.exists(train_csv) or not os.path.exists(val_csv):
            print(f"  [Warning] Missing CSV for Fold {fold}. Skipping.")
            continue

        df_train = pd.read_csv(train_csv)
        df_val = pd.read_csv(val_csv)

        # Feature columns (drop metadata)
        drop_cols = [c for c in META_COLS if c in df_train.columns]
        df_tr_num = df_train.select_dtypes(include=[np.number])
        df_vl_num = df_val.select_dtypes(include=[np.number])
        feat_cols = [c for c in df_tr_num.columns if c not in drop_cols]

        x_tr_flat = df_tr_num[feat_cols].values.astype(np.float32)
        x_vl_flat = df_vl_num[[c for c in df_vl_num.columns if c not in drop_cols]].values.astype(np.float32)
        y_tr = df_train["Health_Index"].values.astype(np.float32)
        y_vl = df_val["Health_Index"].values.astype(np.float32)

        # Pre-training sanity check
        run_sanity_check(feat_cols, x_tr_flat, x_vl_flat,
                         os.path.join(fold_dir, "sanity_check.json"))

        n_feat = x_tr_flat.shape[1] // ws
        print(f"  [INFO] Derived features/timestep: {n_feat} (expected {NUM_FEATURES})")

        # 3D reconstruction
        x_tr_3d = reconstruct_3d(x_tr_flat, ws, n_feat)
        x_vl_3d = reconstruct_3d(x_vl_flat, ws, n_feat)

        # Tensors
        X_train_tensor = torch.tensor(x_tr_3d, dtype=torch.float32).to(DEVICE)
        y_train_tensor = torch.tensor(y_tr, dtype=torch.float32).view(-1, 1).to(DEVICE)
        X_val_tensor = torch.tensor(x_vl_3d, dtype=torch.float32).to(DEVICE)
        y_val_tensor = torch.tensor(y_vl, dtype=torch.float32).view(-1, 1).to(DEVICE)

        train_loader = DataLoader(
            TensorDataset(X_train_tensor, y_train_tensor),
            batch_size=BATCH_SIZE, shuffle=True,
        )

        model = RobustLSTM(
            input_size=n_feat, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
            dropout=DROPOUT, noise_std=NOISE_STD,
        ).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        criterion = nn.MSELoss()

        # Training
        t0 = time.time()
        model.train()
        for epoch in range(EPOCHS):
            ep_loss = 0.0
            for bx, by in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                optimizer.step()
                ep_loss += loss.item()
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"  Epoch [{epoch+1:3d}/{EPOCHS}] MSE={ep_loss/max(len(train_loader),1):.6f}")
        print(f"  Training time: {time.time()-t0:.1f}s")

        # Batched OOM-safe inference
        model.eval()
        preds = []
        with torch.no_grad():
            for bx in DataLoader(X_val_tensor, batch_size=BATCH_SIZE):
                preds.append(model(bx).cpu().numpy().flatten())
        pred_hi = np.clip(np.concatenate(preds), 0.0, 1.0)

        # Distribution logging
        diag_log = log_pred_distribution(pred_hi, diag_log, fold, ws)

        # Build results DataFrame
        val_res = pd.DataFrame()
        val_res["Bearing_ID"] = df_val["Bearing_ID"].values if "Bearing_ID" in df_val.columns else "Unknown"
        if "Step_Index" in df_val.columns:
            val_res["Step_Index"] = df_val["Step_Index"].values
        elif "Original_Minute" in df_val.columns:
            val_res["Step_Index"] = (df_val["Original_Minute"].values / SUBMINUTE_HOP).astype(int)
        else:
            val_res["Step_Index"] = np.arange(len(df_val))
        val_res["Original_Minute"] = val_res["Step_Index"] * SUBMINUTE_HOP
        if "Change_Point" in df_val.columns:
            val_res["Change_Point"] = df_val["Change_Point"].values
        val_res["True_HI"] = y_vl
        val_res["Predicted_HI"] = pred_hi
        val_res["Fold"] = fold
        val_res["Window_Size"] = ws

        # EMA smoothing per bearing
        smoothed = pd.concat([
            b.sort_values("Step_Index").assign(
                Smoothed_Predicted_HI=lambda d: d["Predicted_HI"].ewm(span=5, adjust=False).mean()
            )
            for _, b in val_res.groupby("Bearing_ID")
        ], ignore_index=True)

        # Metrics
        rmse = calculate_rmse(smoothed["True_HI"], smoothed["Smoothed_Predicted_HI"])
        mae = mean_absolute_error(smoothed["True_HI"], smoothed["Smoothed_Predicted_HI"])
        r2 = r2_score(smoothed["True_HI"], smoothed["Smoothed_Predicted_HI"])
        fold_metrics.append({"Fold": fold, "RMSE": rmse, "MAE": mae, "R2": r2})
        print(f"  [Fold {fold}] RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")

        # Save artifacts
        smoothed.to_csv(os.path.join(fold_dir, f"predictions_ws{ws}_fold{fold}.csv"), index=False)
        torch.save(model.state_dict(), os.path.join(fold_dir, f"lstm_ws{ws}_fold{fold}.pth"))
        plot_health_index(smoothed, ws, fold, fold_dir)

        ws_all_preds = pd.concat([ws_all_preds, smoothed], ignore_index=True)

        # ── Strict OOM Prevention ──────────────────────────────────────────
        del X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, model, optimizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Write diagnostic log for this window size
    with open(os.path.join(ws_results_dir, f"diagnostic_log_ws{ws}.json"), "w") as fh:
        json.dump(diag_log, fh, indent=2)

    if fold_metrics:
        df_fm = pd.DataFrame(fold_metrics)
        df_fm.to_csv(os.path.join(ws_results_dir, f"fold_metrics_ws{ws}.csv"), index=False)
        if not ws_all_preds.empty:
            ws_all_preds.to_csv(os.path.join(ws_results_dir, f"all_predictions_ws{ws}.csv"), index=False)
        avgs = df_fm[["RMSE", "MAE", "R2"]].mean()
        print(f"\n  => WS {ws}: RMSE={avgs.RMSE:.4f}  MAE={avgs.MAE:.4f}  R2={avgs.R2:.4f}")
        summary_metrics.append({"Window_Size": ws, "Mean_RMSE": avgs.RMSE,
                                 "Mean_MAE": avgs.MAE, "Mean_R2": avgs.R2})

## 5. Global Summary and Bar Chart

In [ ]:
if summary_metrics:
    summary_df = pd.DataFrame(summary_metrics)
    summary_df.to_csv(os.path.join(RESULTS_DIR, "LSTM_Global_Window_Comparison.csv"), index=False)
    print("\n" + "="*70)
    print("FINAL SUMMARY ACROSS ALL WINDOW SIZES")
    print("="*70)
    print(summary_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(
        summary_df["Window_Size"].astype(str),
        summary_df["Mean_RMSE"],
        color=sns.color_palette("viridis", len(summary_df)),
        edgecolor="black", linewidth=1.2,
    )
    for bar in bars:
        ax.annotate(f"{bar.get_height():.4f}",
                    (bar.get_x() + bar.get_width() / 2.0, bar.get_height()),
                    ha="center", va="bottom", xytext=(0, 6),
                    textcoords="offset points", fontsize=11, fontweight="bold")
    ax.set_title("Mean Validation RMSE Across Window Sizes (LSTM, 5-Fold CV)", pad=12)
    ax.set_xlabel("Window Size (Timesteps)")
    ax.set_ylabel("Mean Validation RMSE (HI scale 0-1)")
    ax.set_ylim(0, summary_df["Mean_RMSE"].max() * 1.25)
    plt.tight_layout()
    chart_path = os.path.join(RESULTS_DIR, "LSTM_RMSE_Comparison_BarChart.png")
    plt.savefig(chart_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[PLOT] Bar chart saved: {chart_path}")
else:
    print("[Warning] No metrics collected. Check dataset and fold CSV files.")